# 02 - Data Quality Assessment

## Purpose

This notebook performs a formal data-quality review before cleaning and modelling.

The goal is to identify issues that matter in a credit-risk setting:

- Missingness and whether it may carry risk signal
- Duplicate and observation-grain issues
- Target imbalance
- Logical validity of numeric fields
- High-cardinality categorical variables
- Leakage-risk features
- Fairness/proxy-risk features

No rows are dropped in this notebook unless explicitly saved as a diagnostic output. Cleaning decisions happen in Notebook 03.

In [11]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

INTERIM_PATH = PROJECT_ROOT / "data" / "interim" / "credit_risk_merged_interim.csv"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

INTERIM_PATH

WindowsPath('D:/Banking and Finance/Projects/canadian-retail-credit-risk-xai/data/interim/credit_risk_merged_interim.csv')

## Load interim analytical dataset

In [12]:
df = pd.read_csv(INTERIM_PATH, low_memory=False)

overview = {
    "row_count": df.shape[0],
    "column_count": df.shape[1],
    "full_duplicate_row_count": int(df.duplicated().sum()),
    "unique_user_id_count": df["user_id"].nunique(),
    "duplicate_user_id_row_count": int(df["user_id"].duplicated().sum()),
    "record_key_duplicate_count": int(df[["user_id", "record_sequence"]].duplicated().sum()) if "record_sequence" in df.columns else None,
}

overview

{'row_count': 134417,
 'column_count': 25,
 'full_duplicate_row_count': 0,
 'unique_user_id_count': 133752,
 'duplicate_user_id_row_count': 665,
 'record_key_duplicate_count': 0}

## Target distribution

For credit default prediction, imbalance is expected. Accuracy alone will not be a reliable model-selection metric.

In [13]:
target_distribution = (
    df["defaulter"]
    .value_counts(dropna=False)
    .rename_axis("defaulter")
    .reset_index(name="row_count")
)
target_distribution["share"] = target_distribution["row_count"] / len(df)
target_distribution.to_csv(TABLE_DIR / "target_distribution.csv", index=False)
target_distribution

,defaulter,row_count,share
0,0,122264,0.9096
1,1,12153,0.0904


## Missingness diagnostics

In [14]:
missingness = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
        "unique_values": df.nunique(dropna=True).values,
    }
).sort_values(["missing_pct", "missing_count"], ascending=False)

missingness.to_csv(TABLE_DIR / "missingness_summary.csv", index=False)
missingness

,column,dtype,missing_count,missing_pct,unique_values
6,employment_type,object,79686,59.2800,2
7,tier_of_employment,object,79686,59.2800,7
13,married,object,45084,33.5400,2
17,social_profile,object,44846,33.3600,2
18,is_verified,object,33507,24.9300,3
2,amount,float64,27619,20.5500,86157
8,industry,object,4,0.0000,12974
10,work_experience,object,4,0.0000,7
0,user_id,int64,0,0.0000,133752
1,loan_category,object,0,0.0000,7


In [15]:
missingness_by_target = []

for col in df.columns:
    if col == "defaulter":
        continue
    temp = (
        df.groupby("defaulter")[col]
        .apply(lambda s: s.isna().mean())
        .reset_index(name="missing_rate")
    )
    temp["column"] = col
    missingness_by_target.append(temp)

missingness_by_target = pd.concat(missingness_by_target, ignore_index=True)
missingness_by_target = missingness_by_target.pivot(index="column", columns="defaulter", values="missing_rate").reset_index()
missingness_by_target.columns = ["column", "missing_rate_non_default", "missing_rate_default"]
missingness_by_target["absolute_gap"] = (
    missingness_by_target["missing_rate_default"] - missingness_by_target["missing_rate_non_default"]
).abs()
missingness_by_target = missingness_by_target.sort_values("absolute_gap", ascending=False)

missingness_by_target.to_csv(TABLE_DIR / "missingness_by_target.csv", index=False)
missingness_by_target.head(20)

,column,missing_rate_non_default,missing_rate_default,absolute_gap
0,amount,0.1895,0.3665,0.1770
19,tier_of_employment,0.5971,0.5499,0.0472
3,employment_type,0.5971,0.5499,0.0472
17,social_profile,0.3341,0.3289,0.0052
11,married,0.3350,0.3392,0.0041
9,is_verified,0.2491,0.2515,0.0025
23,work_experience,0.0000,0.0000,0.0000
6,industry,0.0000,0.0000,0.0000
7,interest_rate,0.0000,0.0000,0.0000
16,role,0.0000,0.0000,0.0000


## Numeric field profiling

This section highlights skewness, extreme values, and possible value-quality problems. These diagnostics guide cleaning and transformations in Notebook 03 and feature engineering in Notebook 05.

In [16]:
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

numeric_profile = df[numeric_cols].agg(["count", "mean", "median", "std", "min", "max"]).T.reset_index()
numeric_profile = numeric_profile.rename(columns={"index": "column"})
numeric_profile["missing_count"] = df[numeric_cols].isna().sum().values
numeric_profile["missing_pct"] = (df[numeric_cols].isna().mean().values * 100).round(2)
numeric_profile["skew"] = df[numeric_cols].skew(numeric_only=True).values

numeric_profile.to_csv(TABLE_DIR / "numeric_profile.csv", index=False)
numeric_profile

,column,count,mean,median,std,min,max,missing_count,missing_pct,skew
0,user_id,"134,417.0000","38,978,883.1536","58,718,437.0000","31,684,296.5972","208,036.0000","78,958,941.0000",0,0.0000,-0.1710
1,amount,"106,798.0000","137,629.1680","76,503.0000","157,596.6500",0.0000,"8,000,078.0000",27619,20.5500,4.3479
2,interest_rate,"134,417.0000",12.0248,11.8400,3.8793,5.4200,23.5400,0,0.0000,0.3644
3,tenure_years,"134,417.0000",4.5231,4.0000,0.8789,4.0000,6.0000,0,0.0000,1.0853
4,record_sequence,"134,417.0000",1.0049,1.0000,0.0702,1.0000,2.0000,0,0.0000,14.1117
5,total_income_pa,"134,417.0000","72,597.9750","62,000.0000","56,100.0513","4,000.0000","7,141,778.0000",0,0.0000,28.7385
6,dependents,"134,417.0000",2.0021,2.0000,1.4140,0.0000,4.0000,0,0.0000,-0.0046
7,delinq_2yrs,"134,417.0000",0.2831,0.0000,0.7992,0.0000,22.0000,0,0.0000,5.4235
8,total_payment,"134,417.0000","10,696.4361","8,061.1400","8,544.3116",0.0000,"57,777.5799",0,0.0000,1.6001
9,received_principal,"134,417.0000","8,282.6762","5,869.1200","7,184.0165",0.0000,"35,000.0100",0,0.0000,1.5556


## Logical data-quality checks

In [17]:
logical_checks = []

def add_check(check_name, condition):
    logical_checks.append(
        {
            "check": check_name,
            "failed_row_count": int(condition.fillna(False).sum()) if hasattr(condition, "fillna") else int(condition.sum()),
            "failed_row_pct": float((condition.fillna(False).mean() * 100)) if hasattr(condition, "fillna") else float(condition.mean() * 100),
        }
    )

add_check("amount_missing", df["amount"].isna())
add_check("amount_non_positive", df["amount"] <= 0)
add_check("interest_rate_outside_0_to_100", (df["interest_rate"] <= 0) | (df["interest_rate"] > 100))
add_check("tenure_years_non_positive", df["tenure_years"] <= 0)
add_check("income_non_positive", df["total_income_pa"] <= 0)
add_check("dependents_negative", df["dependents"] < 0)
add_check("payment_negative", df["total_payment"] < 0)
add_check("received_principal_negative", df["received_principal"] < 0)
add_check("interest_received_negative", df["interest_received"] < 0)
add_check("number_of_loans_negative", df["number_of_loans"] < 0)
add_check(
    "received_principal_greater_than_amount_when_amount_present",
    (df["received_principal"] > df["amount"]) & df["amount"].notna(),
)

logical_quality_checks = pd.DataFrame(logical_checks).sort_values("failed_row_count", ascending=False)
logical_quality_checks.to_csv(TABLE_DIR / "logical_quality_checks.csv", index=False)
logical_quality_checks

,check,failed_row_count,failed_row_pct
0,amount_missing,27619,20.5473
10,received_principal_greater_than_amount_when_am...,2815,2.0942
1,amount_non_positive,19,0.0141
2,interest_rate_outside_0_to_100,0,0.0000
3,tenure_years_non_positive,0,0.0000
4,income_non_positive,0,0.0000
5,dependents_negative,0,0.0000
6,payment_negative,0,0.0000
7,received_principal_negative,0,0.0000
8,interest_received_negative,0,0.0000


## Categorical field profiling

In [18]:
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

categorical_profile = pd.DataFrame(
    {
        "column": categorical_cols,
        "missing_count": [df[col].isna().sum() for col in categorical_cols],
        "missing_pct": [(df[col].isna().mean() * 100).round(2) for col in categorical_cols],
        "unique_values": [df[col].nunique(dropna=True) for col in categorical_cols],
        "top_value": [df[col].mode(dropna=True).iloc[0] if not df[col].mode(dropna=True).empty else np.nan for col in categorical_cols],
        "top_value_share": [
            (df[col].value_counts(dropna=True, normalize=True).iloc[0] * 100).round(2)
            if not df[col].value_counts(dropna=True).empty
            else np.nan
            for col in categorical_cols
        ],
    }
).sort_values("unique_values", ascending=False)

categorical_profile.to_csv(TABLE_DIR / "categorical_profile.csv", index=False)
categorical_profile

,column,missing_count,missing_pct,unique_values,top_value,top_value_share
3,industry,4,0.0000,12974,0,84.5000
9,pincode,0,0.0000,844,XX112X,1.1800
4,role,0,0.0000,46,KHMbckjadbckIFGCASEWdkcndwkcnCCM,15.5300
0,loan_category,0,0.0000,7,Consolidation,60.1600
2,tier_of_employment,79686,59.2800,7,B,30.8600
5,work_experience,4,0.0000,7,0,84.5000
8,home,0,0.0000,5,mortgage,48.7200
6,gender,0,0.0000,3,Female,33.4000
11,is_verified,33507,24.9300,3,Not Verified,33.4200
1,employment_type,79686,59.2800,2,Salaried,81.8900


In [19]:
for col in categorical_cols:
    print(f"\n{col}")
    display(df[col].value_counts(dropna=False).head(10).to_frame("row_count"))


loan_category


,row_count
loan_category,
Consolidation,80868
Credit Card,31668
Home,8503
Other,8225
Business,2236
Car,1538
Medical,1379



employment_type


,row_count
employment_type,
NaN,79686
Salaried,44817
Self - Employeed,9914



tier_of_employment


...

,row_count
pincode,
XX112X,1584
XX945X,1544
XX750X,1501
XX606X,1366
XX100X,1312
XX331X,1275
XX900X,1234
XX70X,1182
XX300X,1145



social_profile


,row_count
social_profile,
No,44912
NaN,44846
Yes,44659



is_verified


,row_count
is_verified,
Not Verified,33722
Source Verified,33630
Verified,33558
NaN,33507


## Leakage and fairness/proxy review

These are not automatic exclusion decisions. They are governance flags to be reviewed before modelling.

- **Leakage risk:** variables that may be known only after loan performance starts or may directly encode repayment/default behaviour.
- **Fairness/proxy risk:** variables that may represent protected characteristics or proxies for protected/social attributes.

In [20]:
leakage_review = pd.DataFrame(
    [
        {"feature": "delinq_2yrs", "risk_level": "medium", "reason": "Historical delinquency may be valid for existing-borrower early warning, but must match prediction timing."},
        {"feature": "total_payment", "risk_level": "high", "reason": "Repayment amount may include post-origination performance and can leak target outcome."},
        {"feature": "received_principal", "risk_level": "high", "reason": "Principal received may be observed after loan performance begins."},
        {"feature": "interest_received", "risk_level": "high", "reason": "Interest received may be post-outcome repayment behaviour."},
        {"feature": "number_of_loans", "risk_level": "medium", "reason": "Could be valid exposure information, but timing must be confirmed."},
        {"feature": "defaulter", "risk_level": "target", "reason": "Target variable; never used as predictor."},
    ]
)

fairness_proxy_review = pd.DataFrame(
    [
        {"feature": "gender", "risk_level": "high", "reason": "Sensitive/protected characteristic; exclude from baseline model and use only for fairness diagnostics if appropriate."},
        {"feature": "married", "risk_level": "medium", "reason": "Household/social status proxy; requires governance review."},
        {"feature": "dependents", "risk_level": "medium", "reason": "Family status proxy; may need careful treatment."},
        {"feature": "home", "risk_level": "medium", "reason": "Wealth and socioeconomic proxy; common in credit risk but should be monitored."},
        {"feature": "pincode", "risk_level": "high", "reason": "Geographic proxy; can encode socioeconomic patterns and requires fairness review."},
        {"feature": "social_profile", "risk_level": "high", "reason": "Unclear meaning and potential social/behavioural proxy; exclude unless business definition is documented."},
        {"feature": "is_verified", "risk_level": "medium", "reason": "Potential operational/verification bias; may be valid but requires interpretation."},
    ]
)

leakage_review.to_csv(TABLE_DIR / "leakage_review.csv", index=False)
fairness_proxy_review.to_csv(TABLE_DIR / "fairness_proxy_review.csv", index=False)

display(leakage_review)
display(fairness_proxy_review)

,feature,risk_level,reason
0,delinq_2yrs,medium,Historical delinquency may be valid for existi...
1,total_payment,high,Repayment amount may include post-origination ...
2,received_principal,high,Principal received may be observed after loan ...
3,interest_received,high,Interest received may be post-outcome repaymen...
4,number_of_loans,medium,"Could be valid exposure information, but timin..."
5,defaulter,target,Target variable; never used as predictor.


,feature,risk_level,reason
0,gender,high,Sensitive/protected characteristic; exclude fr...
1,married,medium,Household/social status proxy; requires govern...
2,dependents,medium,Family status proxy; may need careful treatment.
3,home,medium,Wealth and socioeconomic proxy; common in cred...
4,pincode,high,Geographic proxy; can encode socioeconomic pat...
5,social_profile,high,Unclear meaning and potential social/behaviour...
6,is_verified,medium,Potential operational/verification bias; may b...


## Key Findings:

| Area | Finding | Decision |
|---|---|---|
| Target imbalance | Default rate is about 9.04% | Accuracy alone should not drive model selection |
| Loan amount | 20.55% missing | Keep rows and create a missing-value flag |
| Employment fields | `employment_type` and `tier_of_employment` are 59.28% missing | Do not drop; missingness may carry risk signal |
| Marital, social, and verification fields | 25–34% missing | Create missing-value flags, then fill missing values as `Unknown` |
| Industry and work experience | Many values are placeholder `0` | Treat as unavailable information |
| Repayment fields | `total_payment`, `received_principal`, and `interest_received` | Useful for monitoring, but risky for default modelling due to leakage |
| Sensitive or proxy fields | `gender`, `married`, `pincode`, and `social_profile` | Keep for audit and fairness review, but exclude from the baseline model |

## Data-quality decision log

Use the outputs above to carry these decisions into Notebook 03:

1. Preserve `record_sequence`; do not merge or model on duplicated `user_id` alone.
2. Do not drop employment missingness blindly; missingness is material and may carry risk signal.
3. Treat missing `amount` carefully; create a missingness flag before imputation.
4. Review zero/non-positive loan amounts.
5. Review cases where `received_principal > amount`; this may be legitimate refinance/payment behaviour or an original data-quality issue.
6. Exclude or quarantine high-leakage repayment variables from the first clean baseline model.
7. Exclude `gender` from predictive features; reserve sensitive/proxy attributes for fairness diagnostics and governance discussion.